In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvcc --version


In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126


In [ ]:
!pip install uv

In [ ]:
!uv pip install surya-ocr


In [ ]:
!uv pip install pypdfium2

# extract word

In [ ]:
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
import pypdfium2 as pdfium
import os


In [ ]:
# Initialize predictors once (outside the loop for efficiency)
foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()

In [ ]:
!apt-get update
!apt-get install -y libreoffice


In [ ]:
import os
import shutil
import subprocess
import pypdfium2 as pdfium


# Paths
input_folder = "/content/drive/MyDrive/Graduation Project/Mazen Data/Word"
output_folder = "/content/drive/MyDrive/Graduation Project/Collected Data"
temp_folder = "/content/temp_pdf"
moved_folder = "/content/drive/MyDrive/Graduation Project/Mazen Data/4"

os.makedirs(temp_folder, exist_ok=True)

# Accepted Word formats
extensions = (".docx", ".doc", ".DOC", ".rtf")

docx_files = [f for f in os.listdir(input_folder) if f.endswith(extensions)]
print(f"Found {len(docx_files)} word files to process\n")


# ---------- Convert ANY Word file → PDF using LibreOffice ----------
def convert_to_pdf(input_path):
    filename = os.path.basename(input_path)
    pdf_name = os.path.splitext(filename)[0] + ".pdf"
    output_pdf_path = os.path.join(temp_folder, pdf_name)

    command = [
        "soffice",
        "--headless",
        "--convert-to", "pdf",
        "--outdir", temp_folder,
        input_path
    ]

    subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    return output_pdf_path


# ---------- SURYA OCR ----------
def process_pdf(pdf_path, output_path):
    try:
        pdf = pdfium.PdfDocument(pdf_path)
        n_pages = len(pdf)

        print(f"Total Pages: {n_pages}")
        print("Starting OCR text extraction...\n")

        all_text = []

        for page_num in range(n_pages):
            try:
                print(f"  Processing page {page_num + 1}/{n_pages}...", end=" ")

                page = pdf[page_num]
                pil_image = page.render(scale=1, rotation=0).to_pil()

                detections = detection_predictor([pil_image])
                detected_boxes = detections[0].bboxes

                if len(detected_boxes) > 0:
                    predictions = recognition_predictor([pil_image], det_predictor=detection_predictor)
                    page_text_lines = [line.text for line in predictions[0].text_lines]
                else:
                    page_text_lines = []

                all_text.append({
                    "page": page_num + 1,
                    "full_text": "\n".join(page_text_lines)
                })

                print(f"({len(detected_boxes)} boxes, {len(page_text_lines)} lines)")

            except Exception as e:
                print(f"Error: {str(e)}")
                continue

        # Save extracted text
        with open(output_path, "w", encoding="utf-8") as f:
            for page_data in all_text:
                f.write(f"\n{'='*50}\n")
                f.write(f"PAGE {page_data['page']}\n")
                f.write(f"{'='*50}\n")
                f.write(page_data["full_text"])
                f.write("\n")

        print(f"\nOCR complete: {len(all_text)} pages processed")
        print(f"Saved: {output_path}")

    except Exception as e:
        print(f"Error processing PDF: {str(e)}")


# ---------- MAIN LOOP ----------
for idx, filename in enumerate(docx_files, 1):
    print("\n" + "="*60)
    print(f"[{idx}/{len(docx_files)}] Processing: {filename}")
    print("="*60)

    word_path = os.path.join(input_folder, filename)

    # Define output text file path
    output_filename = os.path.splitext(filename)[0] + ".txt"
    output_path = os.path.join(output_folder, output_filename)

    # 1️⃣ Skip if output already exists
    if os.path.exists(output_path):
        print("Output file already exists → Skipping extraction.")
        print("Moving file to processed folder...")
        shutil.move(word_path, moved_folder)
        continue

    print("Converting Word file → PDF (LibreOffice)...")
    pdf_path = convert_to_pdf(word_path)

    process_pdf(pdf_path, output_path)

    # 2️⃣ Move the original Word file after processing
    print("Moving processed Word file to folder 4...")
    shutil.move(word_path, moved_folder)


print("\n" + "="*60)
print("All files processed successfully!")
print("="*60)
